In [1]:
import pandas as pd
import re

In [2]:
# csvをデータフレームへ
df1 = pd.read_csv('geocode_train.csv')
df2 = pd.read_csv('geocode_test.csv')

train1 = pd.read_csv('train.csv')
test1 = pd.read_csv('test.csv')

In [3]:
# 結合
train = pd.concat([train1, df1], axis=1)
test = pd.concat([test1, df2], axis=1)

In [4]:
# 間取りの'+S(納戸)'を除去
train['間取り'] = train['間取り'].str.replace(r'\+S\(納戸\)', '', regex=True)
test['間取り'] = test['間取り'].str.replace(r'\+S\(納戸\)', '', regex=True)

In [5]:
# Trainデータで区と間取りごとの平均賃料を計算
avg_rent_by_area_room = train.groupby(['区', '間取り'])['賃料'].mean().reset_index()
avg_rent_by_area_room.rename(columns={'賃料': '平均賃料'}, inplace=True)

In [6]:
# 平均賃料を追加
train = pd.merge(train, avg_rent_by_area_room, on=['区', '間取り'], how='left')
test = pd.merge(test, avg_rent_by_area_room, on=['区', '間取り'], how='left')

In [7]:
# 平均賃料がブランク（NaN）の行に対して同じ区の平均値を埋める
test['平均賃料'] = test.groupby('区')['平均賃料'].transform(lambda x: x.fillna(x.mean()))

In [8]:
# 平均賃料を整数に丸める
train['平均賃料'] = train['平均賃料'].round().astype(int)
test['平均賃料'] = test['平均賃料'].round().astype(int)

In [9]:
# 緯度と経度を小数点第7位までで揃える
train['緯度'] = train['緯度'].round(7)
train['経度'] = train['経度'].round(7)
test['緯度'] = test['緯度'].round(7)
test['経度'] = test['経度'].round(7)

In [10]:
# 徒歩時間を正規表現で抽出し、最短のものを取得する関数
def extract_min_walk_time(access_str):
    # 正規表現で「徒歩○分」を抽出
    walk_times = re.findall(r'徒歩(\d+)分', access_str)
    # 数字に変換して最小値を返す
    if walk_times:
        return min(map(int, walk_times))
    return None

In [11]:
# アクセスを最短時間のみに
train['アクセス'] = train['アクセス'].apply(extract_min_walk_time)
test['アクセス'] = test['アクセス'].apply(extract_min_walk_time)

In [12]:
# 間取りを数値に変換するマッピング
location_mapping1 = {location: idx for idx, location in enumerate(test['間取り'].unique())}

In [13]:
# 間取りを数値に変換
train['間取り'] = train['間取り'].map(location_mapping1)
test['間取り'] = test['間取り'].map(location_mapping1)

In [14]:
# 築年数を数値に変換する関数
def convert_age(age_str):
    # 正規表現で「年」と「ヶ月」を分離
    match = re.match(r'(\d+)年(\d+)ヶ月', age_str)
    if match:
        years = int(match.group(1))  # 年
        months = int(match.group(2))  # ヶ月
        # 年とヶ月を数値に変換 (1年 = 12ヶ月)
        return years + months / 12
    else:
        # ○年だけの場合
        match = re.match(r'(\d+)年', age_str)
        if match:
            years = int(match.group(1))
            return years
        # ○ヶ月だけの場合
        match = re.match(r'(\d+)ヶ月', age_str)
        if match:
            months = int(match.group(1))
            return months / 12
        return None  # 不正なデータはNoneとする

In [15]:
# 築年数列を変換
train['築年数'] = train['築年数'].apply(convert_age)
test['築年数'] = test['築年数'].apply(convert_age)

In [16]:
# NaNを0に置き換え
train['築年数'] = train['築年数'].fillna(0).astype(int)
test['築年数'] = test['築年数'].fillna(0).astype(int)

In [17]:
# m2を削除して数値に変換
train['面積'] = train['面積'].str.replace('m2', '').astype(float)
test['面積'] = test['面積'].str.replace('m2', '').astype(float)

In [18]:
# 建物構造を数値に変換するマッピング(手動)
location_mapping2 = {
    'RC（鉄筋コンクリート）': 0,
    'SRC（鉄骨鉄筋コンクリート）': 0,  # コンクリート系にまとめる
    '木造': 1,
    '鉄骨造': 2,
    '軽量鉄骨': 2,  # 鉄骨造にまとめる
    'ALC（軽量気泡コンクリート）': 0,  # コンクリート系にまとめる
    'その他': 3,
    'PC（プレキャスト・コンクリート（鉄筋コンクリート））': 0,  # コンクリート系にまとめる
    'HPC（プレキャスト・コンクリート（重量鉄骨））': 0,  # コンクリート系にまとめる
    '鉄筋ブロック': 0,  # コンクリート系にまとめる
    'ブロック': 0  # コンクリート系にまとめる
}

In [19]:
# 建物構造を数値に変換
train['建物構造'] = train['建物構造'].map(location_mapping2)
test['建物構造'] = test['建物構造'].map(location_mapping2)

In [20]:
# 不要な列を削除
train.drop(['id', '所在地', '所在階', '方角', 'バス・トイレ', 'キッチン', '放送・通信', '室内設備', '駐車場', '周辺環境', '契約期間', '整形住所', '区'], axis=1, inplace=True)
test.drop(['id', '所在地', '所在階', '方角', 'バス・トイレ', 'キッチン', '放送・通信', '室内設備', '駐車場', '周辺環境', '契約期間', '整形住所', '区'], axis=1, inplace=True)

In [21]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 31470 entries, 0 to 31469
Data columns (total 9 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   賃料      31470 non-null  int64  
 1   アクセス    31470 non-null  int64  
 2   間取り     31470 non-null  int64  
 3   築年数     31470 non-null  int64  
 4   面積      31470 non-null  float64
 5   建物構造    31470 non-null  int64  
 6   緯度      31470 non-null  float64
 7   経度      31470 non-null  float64
 8   平均賃料    31470 non-null  int64  
dtypes: float64(3), int64(6)
memory usage: 2.2 MB


In [22]:
test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 31262 entries, 0 to 31261
Data columns (total 8 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   アクセス    31262 non-null  int64  
 1   間取り     31262 non-null  int64  
 2   築年数     31262 non-null  int64  
 3   面積      31262 non-null  float64
 4   建物構造    31262 non-null  int64  
 5   緯度      31262 non-null  float64
 6   経度      31262 non-null  float64
 7   平均賃料    31262 non-null  int64  
dtypes: float64(3), int64(5)
memory usage: 1.9 MB


In [23]:
train.to_csv('train1.csv', index=False)
test.to_csv('test1.csv', index=False)